# Data Cleaning

## Oasis Infobyte – Data Analytics Internship

### Task 3: Cleaning Data



## Objective

The objective of this task is to systematically inspect and clean a deliberately messy dataset. The analysis includes identifying data quality issues, handling missing values, removing duplicates, standardizing inconsistent data, correcting data types, detecting outliers, and producing a clean dataset for analysis.


## Importing Libraries


In [220]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

## Loading the Dataset

In [221]:
df = pd.read_csv("../Data/Raw Data/dirty_cafe_sales.csv")

## Initial Data Inspection

In [222]:
df.shape

(10000, 8)

In [223]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [224]:
df.tail()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02
9999,TXN_6170729,Sandwich,3,4.0,12.0,Cash,In-store,2023-11-07


In [225]:
df.dtypes

Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

In [226]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [227]:
df.describe(include="all").T

,count,unique,top,freq
Transaction ID,10000,10000,TXN_1961373,1
Item,9667,10,Juice,1171
Quantity,9862,7,5,2013
Price Per Unit,9821,8,3.0,2429
Total Spent,9827,19,6.0,979
Payment Method,7421,5,Digital Wallet,2291
Location,6735,4,Takeaway,3022
Transaction Date,9841,367,UNKNOWN,159


In [228]:
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [229]:
df.duplicated().sum()

np.int64(0)

## Data Quality Report

The initial data quality assessment identifies missing values, duplicate records, incorrect data types, inconsistent placeholder values, and potential value range anomalies. This report provides a baseline for evaluating the dataset before and after the cleaning process.

In [230]:
df_raw = df.copy()

quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique(dropna=True).values
})

quality_report

,Column,Data Type,Missing Values,Unique Values
0,Transaction ID,str,0,10000
1,Item,str,333,10
2,Quantity,str,138,7
3,Price Per Unit,str,179,8
4,Total Spent,str,173,19
5,Payment Method,str,2579,5
6,Location,str,3265,4
7,Transaction Date,str,159,367


## Standardising Invalid Values

The dataset contains placeholder values such as `ERROR` and `UNKNOWN`. These values do not represent valid observations, so they are converted to missing values before applying the appropriate missing data handling strategy.

In [231]:
df = df.replace(["ERROR", "UNKNOWN"], np.nan)

df.isnull().sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

## Data Type Correction

The numerical transaction fields are converted to numeric data types, while the transaction date is converted to datetime. The Transaction ID remains a string because it is an identifier rather than a numerical measurement.

In [232]:
df["Transaction ID"] = df["Transaction ID"].astype(str)

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")

df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

df.dtypes

Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

## Handling Missing Values

In [233]:
numeric_columns = ["Quantity", "Price Per Unit", "Total Spent"]
categorical_columns = ["Item", "Payment Method", "Location"]

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

df["Transaction Date"] = df["Transaction Date"].fillna(
    df["Transaction Date"].mode()[0]
)
df["Quantity"] = df["Quantity"].astype(int)
df.isnull().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

- Median imputation is used for numerical columns because the median is less sensitive to extreme values. 
- Mode imputation is used for categorical columns because it preserves the most common valid category. 
- Missing transaction dates are filled using the most frequent valid date because only a small number of dates are missing. 
- The Transaction ID column has no missing values and therefore does not require imputation.

## Standardising Categorical Values

Categorical values are standardised by removing unnecessary spaces and applying consistent capitalization. This prevents equivalent categories from being treated as separate values.

In [234]:
df["Item"] = df["Item"].str.strip().str.title()
df["Payment Method"] = df["Payment Method"].str.strip().str.title()
df["Location"] = df["Location"].str.strip().str.title()

df[["Item", "Payment Method", "Location"]].nunique()

Item              8
Payment Method    3
Location          2
dtype: int64

## Duplicate Removal

In [235]:
duplicate_count = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicates identified:", duplicate_count)
print("Duplicates remaining:", df.duplicated().sum())

Duplicates identified: 0
Duplicates remaining: 0


## Value Range Anomalies

In [236]:
range_anomalies = pd.DataFrame({
    "Column": ["Quantity", "Price Per Unit", "Total Spent"],
    "Negative Values": [
        (df["Quantity"] < 0).sum(),
        (df["Price Per Unit"] < 0).sum(),
        (df["Total Spent"] < 0).sum()
    ],
    "Zero Values": [
        (df["Quantity"] == 0).sum(),
        (df["Price Per Unit"] == 0).sum(),
        (df["Total Spent"] == 0).sum()
    ]
})

range_anomalies

,Column,Negative Values,Zero Values
0,Quantity,0,0
1,Price Per Unit,0,0
2,Total Spent,0,0


The range check found no negative or zero values in the numerical transaction fields. Therefore, no range-based corrections were required, and all numerical values were retained.

## Outlier Detection

In [237]:
outlier_report = []

for column in ["Quantity", "Price Per Unit", "Total Spent"]:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = ((df[column] < lower_bound) | (df[column] > upper_bound)).sum()

    outlier_report.append([
        column,
        q1,
        q3,
        lower_bound,
        upper_bound,
        outlier_count
    ])

outlier_report = pd.DataFrame(
    outlier_report,
    columns=[
        "Column",
        "Q1",
        "Q3",
        "Lower Bound",
        "Upper Bound",
        "Outlier Count"
    ]
)

outlier_report

,Column,Q1,Q3,Lower Bound,Upper Bound,Outlier Count
0,Quantity,2.0,4.0,-1.0,7.0,0
1,Price Per Unit,2.0,4.0,-1.0,7.0,0
2,Total Spent,4.0,12.0,-8.0,24.0,259


## Outlier Treatment Decision

The IQR method identified 259 potential outliers in the Total Spent column, while no outliers were detected in Quantity or Price Per Unit. These observations are retained because high transaction amounts may represent legitimate purchases rather than data entry errors. Removing them could result in loss of useful transaction information.

## Before vs After Cleaning Summary

In [238]:
def dtype_accuracy(dataframe):
    checks = [
        pd.api.types.is_string_dtype(dataframe["Transaction ID"]),
        pd.api.types.is_string_dtype(dataframe["Item"]),
        pd.api.types.is_integer_dtype(dataframe["Quantity"]),
        pd.api.types.is_float_dtype(dataframe["Price Per Unit"]),
        pd.api.types.is_float_dtype(dataframe["Total Spent"]),
        pd.api.types.is_string_dtype(dataframe["Payment Method"]),
        pd.api.types.is_string_dtype(dataframe["Location"]),
        pd.api.types.is_datetime64_any_dtype(dataframe["Transaction Date"])
    ]
    return f"{sum(checks)}/{len(checks)}"

before_after = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Missing Values",
        "Duplicate Rows",
        "Data Type Accuracy"
    ],
    "Before Cleaning": [
        len(df_raw),
        df_raw.isnull().sum().sum(),
        df_raw.duplicated().sum(),
        dtype_accuracy(df_raw)
    ],
    "After Cleaning": [
        len(df),
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        dtype_accuracy(df)
    ]
})

before_after

,Metric,Before Cleaning,After Cleaning
0,Total Rows,10000,10000
1,Missing Values,6826,0
2,Duplicate Rows,0,0
3,Data Type Accuracy,4/8,8/8


## Final Validation

In [239]:
final_validation = pd.DataFrame({
    "Check": [
        "Missing Values",
        "Duplicate Rows",
        "Transaction Date Type",
        "Quantity Numeric",
        "Price Per Unit Numeric",
        "Total Spent Numeric"
    ],
    "Result": [
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        pd.api.types.is_datetime64_any_dtype(df["Transaction Date"]),
        pd.api.types.is_numeric_dtype(df["Quantity"]),
        pd.api.types.is_numeric_dtype(df["Price Per Unit"]),
        pd.api.types.is_numeric_dtype(df["Total Spent"])
    ]
})

final_validation

,Check,Result
0,Missing Values,0
1,Duplicate Rows,0
2,Transaction Date Type,True
3,Quantity Numeric,True
4,Price Per Unit Numeric,True
5,Total Spent Numeric,True


The cleaned dataset is validated to confirm that missing values and duplicate rows have been addressed and that the corrected data types are appropriate for analysis.

## Saving Cleaned Dataset

In [240]:
import os 

os.makedirs("../Data/Cleaned",exist_ok=True)

cleaned_path = "../Data/Cleaned/cleaned_cafe_sales.csv"

df.to_csv(cleaned_path,index=False)

print("Dataset Saved Successfully!")

Dataset Saved Successfully!


## Conclusion

The dataset was successfully cleaned by standardising invalid values, handling missing data, correcting data types, standardising categorical fields, checking duplicates and value ranges, and detecting outliers using the IQR method. No duplicate or range anomalies were found. Potential outliers in Total Spent were retained because they may represent legitimate transactions. The cleaned dataset is suitable for further analysis.